# DICOM Utilities
This notebook contains utility functions to read DICOM files and convert them to JPG format.

### Key Differences and Bug Fixes vs `dicom_utils.py`:

1. **Preserving Directory Structure (`get_names`)**:
   - **`dicom_utils.py`**: Returns only the base filename, causing all images from different subfolders to mix in the output.
   - **`dicom_utils.ipynb`**: Returns the relative path of the DICOM files to maintain the original subfolder hierarchy in the output directory.

2. **Handling Corrupted Files (`ValueError`)**:
   - **`dicom_utils.py`**: The pipeline crashes immediately when encountering corrupted `.dcm` files (e.g., pixel lengths mismatch, `260622 bytes` expected `524288 bytes`).
   - **`dicom_utils.ipynb`**: Wraps the read code within a `try-except` block to catch `ValueError`, allowing the code to safely skip bad files (by returning `None`) without interrupting the conversion loop.

3. **Preventing `NoneType` Save Errors (`AttributeError`)**:
   - **Bug**: Once skipped files return `None`, executing `.save()` on them throws `AttributeError: '''NoneType''' object has no attribute '''save'''`.
   - **Fix**: Added an `if final_image is not None:` condition before saving to ensure only successfully converted images are processed.

4. **Undefined Variable in the Execution Loop (`NameError`)**:
   - **Bug**: The execution loop called `convert_dcm_jpg(cdir, name)` using an undefined `cdir` variable (`NameError: name '''cdir''' is not defined`).
   - **Fix**: Replaced the undefined `cdir` with the properly assigned `input_dir`.

In [5]:
import numpy as np
import pydicom
from PIL import Image
import glob
import os
import pydicom.uid
import tensorflow as tf
from matplotlib import pyplot as plt
import cv2

In [6]:
# Function to read dicom files and return a list of the relative paths of the dicom files
def get_names(path):
    names = []
    for root, _, filenames in os.walk(path):
        for filename in filenames:
            _, ext = os.path.splitext(filename)
            if ext.lower() == '.dcm':
                # Get relative path to maintain folder structure
                rel_dir = os.path.relpath(root, path)
                rel_path = filename if rel_dir == '.' else os.path.join(rel_dir, filename)
                names.append(rel_path)
    return names

In [7]:
# Function to convert dicom files to jpg
def convert_dcm_jpg(cdir, name):
    try:
        im = pydicom.dcmread(cdir + '/' + name)
        im = im.pixel_array.astype(float)
        rescale_image = (np.maximum(im, 0)/im.max())*255
        final_image = np.uint8(rescale_image)
        final_image = Image.fromarray(final_image)
        return final_image
    except ValueError as e:
        print(f"Warning: File {name} corrupted, skipped. Error: {e}")
        return None # Or discard the file here


## Example Workflow
**Step 1:** Define input directory and output directory.

**Step 2:** Utilize the function `get_names` to get the names of the dicom files, as a list.

**Step 3:** Utilize the function `convert_dcm_jpg` to convert the dicom files to jpg.

In [8]:
from tqdm import tqdm
import os

# Example of Step 1, 2, and 3:
input_dir = '/media/NAS_R02/USER_PATH/liwei/github_depo/ILD-Segmentation-And-Classification-DL/ILD_DB'
output_dir = '/media/NAS_R02/USER_PATH/liwei/github_depo/ILD-Segmentation-And-Classification-DL/ILD_DB_fig'
os.makedirs(output_dir, exist_ok=True)
names = get_names(input_dir)

# Use tqdm to display a progress bar
for name in tqdm(names, desc="Converting DICOM to JPG"):
    # Replace cdir with input_dir
    final_image = convert_dcm_jpg(input_dir, name)
    
    if final_image is not None:
        if 'mask' in name.lower():
            out_path = os.path.join(output_dir, name + '.png')
        else:
            out_path = os.path.join(output_dir, name + '.jpg')
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        final_image.save(out_path)


Converting DICOM to JPG:   0%|          | 28/15396 [00:00<06:30, 39.33it/s]/media/NAS_R02/USER_PATH/liwei/pip_tmp/ipykernel_1415036/2422639036.py:6: RuntimeWarning: invalid value encountered in true_divide
  rescale_image = (np.maximum(im, 0)/im.max())*255
Converting DICOM to JPG:  89%|████████▉ | 13695/15396 [02:33<00:11, 145.75it/s]

Converting DICOM to JPG: 100%|██████████| 15396/15396 [02:44<00:00, 93.42it/s] 
